# 02 - Spotify Enrichment

**Input:** scraped articles
**Output:** the same rows plus `spotify_*` columns
**Cost:** one API call per row, plus one per *unique* artist

Bandcamp's own pages give text: artist, album, label, genre, location. What
they don't give is cover art, a canonical release date, or a link anyone can
click. Spotify does, and matching the two gives the dashboard and the web app
something that looks like a product rather than a table.

### Why Client Credentials

Only public catalog data is used (`/search` and `/albums`), so this
authenticates with the **Client Credentials flow** — no user login, no scopes,
no redirect URI. One-time setup:

```bash
pip install -r requirements.txt
cp .env.example .env     # then fill in SPOTIFY_CLIENT_ID / SPOTIFY_CLIENT_SECRET
```

Credentials come from <https://developer.spotify.com/dashboard>.

In [ ]:
import sys
from pathlib import Path

# Make the pipeline package importable without installing it, so the notebook
# runs on a fresh clone. `pip install -e .` also works and is preferred.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from bandcamp_aotd.logging_config import setup_logging

setup_logging()

## 1. Load the data to enrich

In [ ]:
import pandas as pd
from bandcamp_aotd import config

df = pd.read_csv(config.ENRICHED_CSV)
print(df.shape)
df.head(3)

## 2. Test on a handful of rows first

Always. This confirms the credentials work and that matching behaves before
spending two thousand API calls finding out it doesn't.

In [ ]:
from bandcamp_aotd.enrich import SpotifyClient

client = SpotifyClient()

sample = df.head(5)
sample_results = client.enrich_dataframe(sample, artist_col="Artist", album_col="Album")

pd.concat([
    sample[["Artist", "Album"]].reset_index(drop=True),
    sample_results[["spotify_match_status", "spotify_match_artist",
                    "spotify_match_album", "spotify_release_date"]].reset_index(drop=True),
], axis=1)

## 3. Run the full album pass

Three things make a 2,300-row run survivable:

**Checkpointing.** Progress is written to CSV every 50 rows. If the kernel
dies, just re-run the cell — completed rows are skipped and previously-errored
rows are retried. Nothing is paid for twice.

**Per-row error capture.** An individual failure lands in
`spotify_match_status` instead of raising, so one weird compilation doesn't
end the run.

**Run-level abort on a hard rate limit.** This one was learned the hard way.
An earlier version of this pipeline paced at ~10 requests/second and Spotify
answered with `Retry-After: 86109` — a 24-hour cooldown. Because the per-row
handler treated it as an ordinary error, it burned through the remaining 1,592
rows in seconds, marked every one of them failed, and poisoned the checkpoint
with work that was never attempted.

A long rate limit is a *run-level* condition, not a row-level one. It now
raises `SpotifyRateLimitError`, which is deliberately not caught per row: the
run stops, the checkpoint is flushed first, and re-running after the cooldown
resumes from exactly the right place. The default pacing is also gentler
(~3 requests/second), which finishes the full dataset in about 13 minutes
without tripping the limit at all.

In [ ]:
from bandcamp_aotd.enrich import SpotifyRateLimitError

try:
    spotify_results = client.enrich_dataframe(
        df,
        artist_col="Artist",
        album_col="Album",
        checkpoint_path=str(config.SPOTIFY_ALBUM_CHECKPOINT),
    )
except SpotifyRateLimitError as exc:
    print(f"Stopped early: {exc}")
    print("Progress is checkpointed - re-run this cell after the cooldown.")
    raise

spotify_results["spotify_match_status"].value_counts()

## 4. Merge and save

In [ ]:
enriched = pd.concat(
    [df.reset_index(drop=True), spotify_results.reset_index(drop=True)], axis=1
)
enriched.to_csv(config.ENRICHED_CSV, index=False)
print(f"Match rate: {enriched['spotify_match_status'].eq('matched').mean():.1%}")
enriched.head(3)

## 5. Enrich artists

One call per **unique** artist rather than per row. `enrich_artists` dedups
automatically, which matters less than it sounds here (most artists appear
once) but costs nothing and makes the pass cheaper as the archive grows.

This adds the artist photo and profile link — the two fields the artist
endpoint provides that aren't already on the album object.

In [ ]:
artist_results = client.enrich_artists(
    enriched["spotify_artist_id"],
    checkpoint_path=str(config.SPOTIFY_ARTIST_CHECKPOINT),
)
artist_results["spotify_artist_status"].value_counts()

In [ ]:
enriched = enriched.merge(
    artist_results, left_on="spotify_artist_id", right_index=True, how="left"
)
enriched.to_csv(config.ENRICHED_CSV, index=False)
enriched.head(3)

## Notes on trusting this data

- **Filter on status before trusting any `spotify_*` column.**
  `spotify_match_status` and `spotify_artist_status` are each `matched`,
  `no_match` / `no_id`, or `error: ...`.

- **Spot-check for false positives.** Spotify's search relevance and Bandcamp's
  naming don't always agree, especially for compilations and split releases.
  Compare `spotify_match_artist` against `Artist` on a sample.

- **A `no_match` is a finding, not a gap.** Plenty of Bandcamp Daily picks
  simply aren't on Spotify — that's part of what makes Bandcamp Bandcamp. The
  unmatched rate is worth reporting, not hiding.

- **Deliberately omitted:** `Album.genres`, `Album.label`, `Album.popularity`
  and the artist equivalents are deprecated in Spotify's current API and come
  back empty or unreliable. The Bandcamp `genre_tag` and `record_label` columns
  are the better source and are already in the dataset.